In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "71c87f1c",
   "metadata": {},
   "source": [
    "# Naive Bayes"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 11,
   "id": "5ac84545",
   "metadata": {
    "language_info": {
     "name": "polyglot-notebook"
    },
    "polyglot_notebook": {
     "kernelName": "csharp"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div><div></div><div></div><div><strong>Installed Packages</strong><ul><li><span>Sep, 0.12.1</span></li></ul></div></div>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "#r \"nuget: Sep, 0.12.1\"\n",
    "\n",
    "using nietras.SeparatedValues;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 12,
   "id": "ebada3ac",
   "metadata": {
    "language_info": {
     "name": "polyglot-notebook"
    },
    "polyglot_notebook": {
     "kernelName": "csharp"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div class=\"dni-plaintext\"><pre>[ enron, methanol, meter, is, a, follow, up, to, the, note, i, gave, you, on, monday, data, provided, by, daren, override ... (21 more) ]</pre></div><style>\r\n",
       ".dni-code-hint {\r\n",
       "    font-style: italic;\r\n",
       "    overflow: hidden;\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview {\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview td {\r\n",
       "    vertical-align: top;\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "details.dni-treeview {\r\n",
       "    padding-left: 1em;\r\n",
       "}\r\n",
       "table td {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "table tr { \r\n",
       "    vertical-align: top; \r\n",
       "    margin: 0em 0px;\r\n",
       "}\r\n",
       "table tr td pre \r\n",
       "{ \r\n",
       "    vertical-align: top !important; \r\n",
       "    margin: 0em 0px !important;\r\n",
       "} \r\n",
       "table th {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "</style>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "\n",
    "record SpamDataSet(int Id, bool Spam, string Text, string[] Words);\n",
    "\n",
    "\n",
    "IEnumerable<string> GetWords(string text) \n",
    "{\n",
    "    return text.Split(\" \").Where(x => x.All(char.IsAsciiLetter));\n",
    "    // return text.Split(\" \");\n",
    "}\n",
    "\n",
    "IEnumerable<SpamDataSet> ReadDatabase() {\n",
    "    using var reader = Sep.Reader().FromFile(\"spam_ham_dataset.csv\");\n",
    "\n",
    "    foreach (var row in reader) {\n",
    "        var text = row[\"text\"].ToString();\n",
    "\n",
    "        yield return new SpamDataSet(\n",
    "            row[\"\"].Parse<int>(),\n",
    "            row[\"label\"].ToString() == \"spam\",\n",
    "            text,\n",
    "            GetWords(text).ToArray()\n",
    "        );\n",
    "    }\n",
    "\n",
    "}\n",
    "\n",
    "\n",
    "var rawData = ReadDatabase().ToArray();\n",
    "rawData.First().Words"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 13,
   "id": "8356f116",
   "metadata": {
    "language_info": {
     "name": "polyglot-notebook"
    },
    "polyglot_notebook": {
     "kernelName": "csharp"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div class=\"dni-plaintext\"><pre>[ enron, methanol, meter, is, a, follow, up, to, the, note, i, gave, you, on, monday, data, provided, by, daren, override ... (40472 more) ]</pre></div><style>\r\n",
       ".dni-code-hint {\r\n",
       "    font-style: italic;\r\n",
       "    overflow: hidden;\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview {\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview td {\r\n",
       "    vertical-align: top;\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "details.dni-treeview {\r\n",
       "    padding-left: 1em;\r\n",
       "}\r\n",
       "table td {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "table tr { \r\n",
       "    vertical-align: top; \r\n",
       "    margin: 0em 0px;\r\n",
       "}\r\n",
       "table tr td pre \r\n",
       "{ \r\n",
       "    vertical-align: top !important; \r\n",
       "    margin: 0em 0px !important;\r\n",
       "} \r\n",
       "table th {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "</style>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "var bagOfWords = rawData.SelectMany(x => x.Words).Distinct().ToArray();\n",
    "bagOfWords"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 14,
   "id": "5a3dbf70",
   "metadata": {
    "language_info": {
     "name": "polyglot-notebook"
    },
    "polyglot_notebook": {
     "kernelName": "csharp"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<details open=\"open\" class=\"dni-treeview\"><summary><span class=\"dni-code-hint\"><code>(5171, 5171)</code></span></summary><div><table><thead><tr></tr></thead><tbody><tr><td>Item1</td><td><div class=\"dni-plaintext\"><pre>5171</pre></div></td></tr><tr><td>Item2</td><td><div class=\"dni-plaintext\"><pre>5171</pre></div></td></tr></tbody></table></div></details><style>\r\n",
       ".dni-code-hint {\r\n",
       "    font-style: italic;\r\n",
       "    overflow: hidden;\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview {\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview td {\r\n",
       "    vertical-align: top;\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "details.dni-treeview {\r\n",
       "    padding-left: 1em;\r\n",
       "}\r\n",
       "table td {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "table tr { \r\n",
       "    vertical-align: top; \r\n",
       "    margin: 0em 0px;\r\n",
       "}\r\n",
       "table tr td pre \r\n",
       "{ \r\n",
       "    vertical-align: top !important; \r\n",
       "    margin: 0em 0px !important;\r\n",
       "} \r\n",
       "table th {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "</style>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "static Random rng = new Random(420);\n",
    "\n",
    "static void Shuffle<T>(this IList<T> list)  \n",
    "{  \n",
    "    int n = list.Count;  \n",
    "    while (n > 1) {  \n",
    "        n--;  \n",
    "        int k = rng.Next(n + 1);  \n",
    "        T value = list[k];  \n",
    "        list[k] = list[n];  \n",
    "        list[n] = value;  \n",
    "    }  \n",
    "}\n",
    "\n",
    "rawData.Shuffle();\n",
    "var splitIndex = (int)(rawData.Length * 0.8);\n",
    "\n",
    "var train = rawData.Take(splitIndex).ToArray();\n",
    "var test = rawData.Skip(splitIndex).ToArray();\n",
    "\n",
    "(train.Length + test.Length, rawData.Length)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 15,
   "id": "718018c1",
   "metadata": {
    "language_info": {
     "name": "polyglot-notebook"
    },
    "polyglot_notebook": {
     "kernelName": "csharp"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<table><thead><tr><th><i>index</i></th><th>value</th></tr></thead><tbody><tr><td>0</td><td><details class=\"dni-treeview\"><summary><span class=\"dni-code-hint\"><code>[enron, (4100 / 0 / 4100)]</code></span></summary><div><table><thead><tr></tr></thead><tbody><tr><td>Key</td><td><div class=\"dni-plaintext\"><pre>enron</pre></div></td></tr><tr><td>Value</td><td><details class=\"dni-treeview\"><summary><span class=\"dni-code-hint\"><code>(4100 / 0 / 4100)</code></span></summary><div><table><thead><tr></tr></thead><tbody><tr><td>Count</td><td><div class=\"dni-plaintext\"><pre>4100</pre></div></td></tr><tr><td>Spam</td><td><div class=\"dni-plaintext\"><pre>0</pre></div></td></tr><tr><td>Ham</td><td><div class=\"dni-plaintext\"><pre>4100</pre></div></td></tr></tbody></table></div></details></td></tr></tbody></table></div></details></td></tr><tr><td>1</td><td><details class=\"dni-treeview\"><summary><span class=\"dni-code-hint\"><code>[methanol, (62 / 0 / 62)]</code></span></summary><div><table><thead><tr></tr></thead><tbody><tr><td>Key</td><td><div class=\"dni-plaintext\"><pre>methanol</pre></div></td></tr><tr><td>Value</td><td><details class=\"dni-treeview\"><summary><span class=\"dni-code-hint\"><code>(62 / 0 / 62)</code></span></summary><div><table><thead><tr></tr></thead><tbody><tr><td>Count</td><td><div class=\"dni-plaintext\"><pre>62</pre></div></td></tr><tr><td>Spam</td><td><div class=\"dni-plaintext\"><pre>0</pre></div></td></tr><tr><td>Ham</td><td><div class=\"dni-plaintext\"><pre>62</pre></div></td></tr></tbody></table></div></details></td></tr></tbody></table></div></details></td></tr><tr><td>2</td><td><details class=\"dni-treeview\"><summary><span class=\"dni-code-hint\"><code>[meter, (1692 / 0 / 1692)]</code></span></summary><div><table><thead><tr></tr></thead><tbody><tr><td>Key</td><td><div class=\"dni-plaintext\"><pre>meter</pre></div></td></tr><tr><td>Value</td><td><details class=\"dni-treeview\"><summary><span class=\"dni-code-hint\"><code>(1692 / 0 / 1692)</code></span></summary><div><table><thead><tr></tr></thead><tbody><tr><td>Count</td><td><div class=\"dni-plaintext\"><pre>1692</pre></div></td></tr><tr><td>Spam</td><td><div class=\"dni-plaintext\"><pre>0</pre></div></td></tr><tr><td>Ham</td><td><div class=\"dni-plaintext\"><pre>1692</pre></div></td></tr></tbody></table></div></details></td></tr></tbody></table></div></details></td></tr><tr><td>3</td><td><details class=\"dni-treeview\"><summary><span class=\"dni-code-hint\"><code>[is, (5217 / 1683 / 3534)]</code></span></summary><div><table><thead><tr></tr></thead><tbody><tr><td>Key</td><td><div class=\"dni-plaintext\"><pre>is</pre></div></td></tr><tr><td>Value</td><td><details class=\"dni-treeview\"><summary><span class=\"dni-code-hint\"><code>(5217 / 1683 / 3534)</code></span></summary><div><table><thead><tr></tr></thead><tbody><tr><td>Count</td><td><div class=\"dni-plaintext\"><pre>5217</pre></div></td></tr><tr><td>Spam</td><td><div class=\"dni-plaintext\"><pre>1683</pre></div></td></tr><tr><td>Ham</td><td><div class=\"dni-plaintext\"><pre>3534</pre></div></td></tr></tbody></table></div></details></td></tr></tbody></table></div></details></td></tr><tr><td>4</td><td><details class=\"dni-treeview\"><summary><span class=\"dni-code-hint\"><code>[a, (7048 / 2665 / 4383)]</code></span></summary><div><table><thead><tr></tr></thead><tbody><tr><td>Key</td><td><div class=\"dni-plaintext\"><pre>a</pre></div></td></tr><tr><td>Value</td><td><details class=\"dni-treeview\"><summary><span class=\"dni-code-hint\"><code>(7048 / 2665 / 4383)</code></span></summary><div><table><thead><tr></tr></thead><tbody><tr><td>Count</td><td><div class=\"dni-plaintext\"><pre>7048</pre></div></td></tr><tr><td>Spam</td><td><div class=\"dni-plaintext\"><pre>2665</pre></div></td></tr><tr><td>Ham</td><td><div class=\"dni-plaintext\"><pre>4383</pre></div></td></tr></tbody></table></div></details></td></tr></tbody></table></div></details></td></tr></tbody></table><style>\r\n",
       ".dni-code-hint {\r\n",
       "    font-style: italic;\r\n",
       "    overflow: hidden;\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview {\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview td {\r\n",
       "    vertical-align: top;\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "details.dni-treeview {\r\n",
       "    padding-left: 1em;\r\n",
       "}\r\n",
       "table td {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "table tr { \r\n",
       "    vertical-align: top; \r\n",
       "    margin: 0em 0px;\r\n",
       "}\r\n",
       "table tr td pre \r\n",
       "{ \r\n",
       "    vertical-align: top !important; \r\n",
       "    margin: 0em 0px !important;\r\n",
       "} \r\n",
       "table th {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "</style>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "class WordCounter {\n",
    "\n",
    "    public int Count {get; private set;} = 0;\n",
    "    public int Spam {get; private set;} = 0;\n",
    "    public int Ham => Count - Spam;\n",
    "\n",
    "    public void AddCount(bool spam) {\n",
    "        Count++;\n",
    "        if (spam)\n",
    "            Spam++;\n",
    "    }\n",
    "\n",
    "    public override string ToString() => $\"({Count} / {Spam} / {Ham})\";\n",
    "}\n",
    "\n",
    "Dictionary<string, WordCounter> counters = bagOfWords.ToDictionary(x => x, _ => new WordCounter());\n",
    "\n",
    "foreach (var r in train) {\n",
    "    foreach (var word in r.Words) counters[word].AddCount(r.Spam);\n",
    "}\n",
    "\n",
    "counters.Take(5)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "2f6ddaf7",
   "metadata": {
    "language_info": {
     "name": "polyglot-notebook"
    },
    "polyglot_notebook": {
     "kernelName": "csharp"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<details open=\"open\" class=\"dni-treeview\"><summary><span class=\"dni-code-hint\"><code>(0.28820116054158607, 0.7117988394584139)</code></span></summary><div><table><thead><tr></tr></thead><tbody><tr><td>Item1</td><td><div class=\"dni-plaintext\"><pre>0.28820116054158607</pre></div></td></tr><tr><td>Item2</td><td><div class=\"dni-plaintext\"><pre>0.7117988394584139</pre></div></td></tr></tbody></table></div></details><style>\r\n",
       ".dni-code-hint {\r\n",
       "    font-style: italic;\r\n",
       "    overflow: hidden;\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview {\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview td {\r\n",
       "    vertical-align: top;\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "details.dni-treeview {\r\n",
       "    padding-left: 1em;\r\n",
       "}\r\n",
       "table td {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "table tr { \r\n",
       "    vertical-align: top; \r\n",
       "    margin: 0em 0px;\r\n",
       "}\r\n",
       "table tr td pre \r\n",
       "{ \r\n",
       "    vertical-align: top !important; \r\n",
       "    margin: 0em 0px !important;\r\n",
       "} \r\n",
       "table th {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "</style>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "double SpamCount = rawData.Count(x => x.Spam);\n",
    "double HamCount = rawData.Count(x => !x.Spam);\n",
    "\n",
    "// double PSpam(WordCounter wc) => (double)wc.Spam / (double)wc.Count;\n",
    "// double PHam(WordCounter wc) => (double)wc.Ham / (double)wc.Count;\n",
    "\n",
    "double PSpam(WordCounter wc) => (double)wc.Spam / (double)SpamCount;\n",
    "double PHam(WordCounter wc) => (double)wc.Ham / (double)HamCount;\n",
    "\n",
    "// double PSpamL(WordCounter wc, double m = 1) => ((double)wc.Spam + m * 1/(double)wc.Count) / ((double)wc.Count + m);\n",
    "// double PHamL(WordCounter wc, double m = 1) => ((double)wc.Ham + m * 1/(double)wc.Count) / ((double)wc.Count + m);\n",
    "\n",
    "// Laplace\n",
    "double PSpamL(WordCounter wc, double m = 1) => ((double)wc.Spam + m * 1/(double)SpamCount) / (SpamCount + m * (double)wc.Count);\n",
    "double PHamL(WordCounter wc, double m = 1) => ((double)wc.Ham + m * 1/(double)HamCount) / (HamCount + m * (double)wc.Count);\n",
    "\n",
    "double PSpam(ICollection<SpamDataSet> ds) => (double)ds.Count(x => x.Spam) / (double)ds.Count;\n",
    "double PHam(ICollection<SpamDataSet> ds) => (double)ds.Count(x => !x.Spam) / (double)ds.Count;\n",
    "\n",
    "(PSpam(train), PHam(train))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 17,
   "id": "8020df59",
   "metadata": {
    "language_info": {
     "name": "polyglot-notebook"
    },
    "polyglot_notebook": {
     "kernelName": "csharp"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<details open=\"open\" class=\"dni-treeview\"><summary><span class=\"dni-code-hint\"><code>(0.744927536231884, 0.9429400386847195)</code></span></summary><div><table><thead><tr></tr></thead><tbody><tr><td>Item1</td><td><div class=\"dni-plaintext\"><pre>0.744927536231884</pre></div></td></tr><tr><td>Item2</td><td><div class=\"dni-plaintext\"><pre>0.9429400386847195</pre></div></td></tr></tbody></table></div></details><style>\r\n",
       ".dni-code-hint {\r\n",
       "    font-style: italic;\r\n",
       "    overflow: hidden;\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview {\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview td {\r\n",
       "    vertical-align: top;\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "details.dni-treeview {\r\n",
       "    padding-left: 1em;\r\n",
       "}\r\n",
       "table td {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "table tr { \r\n",
       "    vertical-align: top; \r\n",
       "    margin: 0em 0px;\r\n",
       "}\r\n",
       "table tr td pre \r\n",
       "{ \r\n",
       "    vertical-align: top !important; \r\n",
       "    margin: 0em 0px !important;\r\n",
       "} \r\n",
       "table th {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "</style>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "var _PSpam = PSpam(train);\n",
    "var _PHam = PHam(train);\n",
    "\n",
    "bool ClassifySpam(string text) {\n",
    "\n",
    "    var pSpam = _PSpam;\n",
    "    var pHam = _PHam;\n",
    "\n",
    "    foreach (var word in GetWords(text)) {\n",
    "        pSpam *= PSpam(counters[word]);\n",
    "        pHam *= PHam(counters[word]);\n",
    "    }\n",
    "\n",
    "    return pSpam > pHam;\n",
    "}\n",
    "\n",
    "var check1 = test.Select(x => (x, ClassifySpam(x.Text))).ToArray();\n",
    "var check2 = train.Select(x => (x, ClassifySpam(x.Text))).ToArray();\n",
    "\n",
    "var correct = check1.Count(x => x.Item1.Spam == x.Item2);\n",
    "var correct2 = check2.Count(x => x.Item1.Spam == x.Item2);\n",
    "\n",
    "((double)correct / (double)check1.Length, (double)correct2 / (double)check2.Length)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 24,
   "id": "fd7a56e6",
   "metadata": {
    "language_info": {
     "name": "polyglot-notebook"
    },
    "polyglot_notebook": {
     "kernelName": "csharp"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div class=\"dni-plaintext\"><pre>0.914975845410628</pre></div><style>\r\n",
       ".dni-code-hint {\r\n",
       "    font-style: italic;\r\n",
       "    overflow: hidden;\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview {\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview td {\r\n",
       "    vertical-align: top;\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "details.dni-treeview {\r\n",
       "    padding-left: 1em;\r\n",
       "}\r\n",
       "table td {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "table tr { \r\n",
       "    vertical-align: top; \r\n",
       "    margin: 0em 0px;\r\n",
       "}\r\n",
       "table tr td pre \r\n",
       "{ \r\n",
       "    vertical-align: top !important; \r\n",
       "    margin: 0em 0px !important;\r\n",
       "} \r\n",
       "table th {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "</style>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "var _PSpam = PSpam(train);\n",
    "var _PHam = PHam(train);\n",
    "\n",
    "bool ClassifySpamL(string text) {\n",
    "\n",
    "    var pSpam = _PSpam;\n",
    "    var pHam = _PHam;\n",
    "\n",
    "    foreach (var word in GetWords(text)) {\n",
    "        pSpam *= PSpamL(counters[word]);\n",
    "        pHam *= PHamL(counters[word]);\n",
    "    }\n",
    "\n",
    "    return pSpam > pHam;\n",
    "}\n",
    "\n",
    "var check1 = test.Select(x => (x, ClassifySpamL(x.Text))).ToArray();\n",
    "\n",
    "var correct = check1.Count(x => x.Item1.Spam == x.Item2);\n",
    "(double)correct / (double)check1.Length"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 19,
   "id": "d11f5c42",
   "metadata": {
    "language_info": {
     "name": "polyglot-notebook"
    },
    "polyglot_notebook": {
     "kernelName": "csharp"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div class=\"dni-plaintext\"><pre>0.7468599033816425</pre></div><style>\r\n",
       ".dni-code-hint {\r\n",
       "    font-style: italic;\r\n",
       "    overflow: hidden;\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview {\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview td {\r\n",
       "    vertical-align: top;\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "details.dni-treeview {\r\n",
       "    padding-left: 1em;\r\n",
       "}\r\n",
       "table td {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "table tr { \r\n",
       "    vertical-align: top; \r\n",
       "    margin: 0em 0px;\r\n",
       "}\r\n",
       "table tr td pre \r\n",
       "{ \r\n",
       "    vertical-align: top !important; \r\n",
       "    margin: 0em 0px !important;\r\n",
       "} \r\n",
       "table th {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "</style>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "var _PSpam = PSpam(train);\n",
    "var _PHam = PHam(train);\n",
    "\n",
    "bool ClassifySpamLog(string text) {\n",
    "\n",
    "    var pSpam = Math.Log(_PSpam);\n",
    "    var pHam = Math.Log(_PHam);\n",
    "\n",
    "    foreach (var word in GetWords(text)) {\n",
    "        pSpam += Math.Log(PSpam(counters[word]));\n",
    "        pHam += Math.Log(PHam(counters[word]));\n",
    "    }\n",
    "\n",
    "    return pSpam > pHam;\n",
    "}\n",
    "\n",
    "var check1 = test.Select(x => (x, ClassifySpamLog(x.Text))).ToArray();\n",
    "\n",
    "var correct = check1.Count(x => x.Item1.Spam == x.Item2);\n",
    "(double)correct / (double)check1.Length"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 20,
   "id": "ccfa1af3",
   "metadata": {
    "language_info": {
     "name": "polyglot-notebook"
    },
    "polyglot_notebook": {
     "kernelName": "csharp"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div class=\"dni-plaintext\"><pre>0.9120772946859903</pre></div><style>\r\n",
       ".dni-code-hint {\r\n",
       "    font-style: italic;\r\n",
       "    overflow: hidden;\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview {\r\n",
       "    white-space: nowrap;\r\n",
       "}\r\n",
       ".dni-treeview td {\r\n",
       "    vertical-align: top;\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "details.dni-treeview {\r\n",
       "    padding-left: 1em;\r\n",
       "}\r\n",
       "table td {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "table tr { \r\n",
       "    vertical-align: top; \r\n",
       "    margin: 0em 0px;\r\n",
       "}\r\n",
       "table tr td pre \r\n",
       "{ \r\n",
       "    vertical-align: top !important; \r\n",
       "    margin: 0em 0px !important;\r\n",
       "} \r\n",
       "table th {\r\n",
       "    text-align: start;\r\n",
       "}\r\n",
       "</style>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "var _PSpam = PSpam(train);\n",
    "var _PHam = PHam(train);\n",
    "\n",
    "bool ClassifySpamLogL(string text) {\n",
    "\n",
    "    var pSpam = Math.Log(_PSpam);\n",
    "    var pHam = Math.Log(_PHam);\n",
    "\n",
    "    foreach (var word in GetWords(text)) {\n",
    "        pSpam += Math.Log(PSpamL(counters[word]));\n",
    "        pHam += Math.Log(PHamL(counters[word]));\n",
    "    }\n",
    "\n",
    "    return pSpam > pHam;\n",
    "}\n",
    "\n",
    "var check1 = test.Select(x => (x, ClassifySpamL(x.Text))).ToArray();\n",
    "\n",
    "var correct = check1.Count(x => x.Item1.Spam == x.Item2);\n",
    "(double)correct / (double)check1.Length"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "language_info": {
     "name": "polyglot-notebook"
    },
    "polyglot_notebook": {
     "kernelName": "csharp"
    }
   },
   "outputs": [],
   "source": []
  },
  {
   "cell_type": "markdown",
   "id": "980c88de",
   "metadata": {},
   "source": [
    "# Post mortem\n",
    "\n",
    "Jede Person beschreibt in der ILIAS-Abgabe individuell(!) die Bearbeitung des jeweiligen Aufgabenblattes\n",
    "zurückblickend mit ca. 200 bis 400 Wörtern. Gehen Sie dabei aussagekräftig und nachvollziehbar auf folgende Punkte ein: \n",
    " (a) Zusammenfassung: Was wurde gemacht?  \n",
    " (b) Implementierungsdetails: Kurze Beschreibung besonders interessanter Aspekte der Umsetzung.  \n",
    " (c) Was war der schwierigste Teil bei der Bearbeitung? Wie haben Sie dieses Problem gelöst?  \n",
    " (d) Was haben Sie gelernt oder (besser) verstanden?  \n",
    " (e) Team: Mit wem haben Sie zusammengearbeitet?  \n",
    " (f) Link zum Repo mit der Lösung  \n",
    "\n",
    "\n",
    "\n",
    " * a: Für die erste Aufgabe habe ich einen Naive Bayes für einen Datensatz aus einem vorherigen Praktikum schriftlich berechnet. Für die zweite Aufgabe habe ich einen Naive Bayes in C# implementiert, um einen E-Mail spamfilter zu erstellen. Die Textqualifikation wurde an einem Beispieldatensatz trainiert.\n",
    " * b: Für die Spam-Erkennung habe ich zu erst aus den Texten die Merkmale erzeugt. Dies habe ich sehr simple gehalten. Ich habe den Text an den Leerzeichen gesplittet und nur die Worte behalten, die Ausschließlich aus Buchstaben bestehen. Aus den Worten aus allen Texten einen \"Bag of Words\" erzeugt. Danach habe ich gezählt, wie häufig welche Worte in welcher Klasse vorkommen. Anhand der Anzahl der Vorkommnisse kann dan die Wahrscheinlichkeit berechnet werden, mit der ein Wort in einer Klasse vor kommt.\n",
    " * c: Ich hatte zu Anfang keinen Blick, wie ich den Naive Bayes überhaupt berechne. Ich habe dan die wichtigen Regeln Aufgeschrieben und mir Kommentare dazu geschrieben. Dadurch habe ich tatsächlich verstanden, wie der Algorithmus funktioniert und das er eigentlich recht einfach ist.\n",
    " * d: \n",
    " * e: -\n",
    " * f: https://github.com/co1inco/IFM_Programieren3/tree/master/KI/10"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": ".NET (C#)",
   "language": "C#",
   "name": ".net-csharp"
  },
  "language_info": {
   "name": "polyglot-notebook"
  },
  "polyglot_notebook": {
   "kernelInfo": {
    "defaultKernelName": "csharp",
    "items": [
     {
      "aliases": [],
      "languageName": "csharp",
      "name": "csharp"
     }
    ]
   }
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}


In [11]:
#r "nuget: Sep, 0.12.1"

using nietras.SeparatedValues;

Installed Packages Sep, 0.12.1

In [12]:

record SpamDataSet(int Id, bool Spam, string Text, string[] Words);


IEnumerable<string> GetWords(string text) 
{
    return text.Split(" ").Where(x => x.All(char.IsAsciiLetter));
    // return text.Split(" ");
}

IEnumerable<SpamDataSet> ReadDatabase() {
    using var reader = Sep.Reader().FromFile("spam_ham_dataset.csv");

    foreach (var row in reader) {
        var text = row["text"].ToString();

        yield return new SpamDataSet(
            row[""].Parse<int>(),
            row["label"].ToString() == "spam",
            text,
            GetWords(text).ToArray()
        );
    }

}


var rawData = ReadDatabase().ToArray();
rawData.First().Words

[ enron, methanol, meter, is, a, follow, up, to, the, note, i, gave, you, on, monday, data, provided, by, daren, override ... (21 more) ]

In [13]:
var bagOfWords = rawData.SelectMany(x => x.Words).Distinct().ToArray();
bagOfWords

[ enron, methanol, meter, is, a, follow, up, to, the, note, i, gave, you, on, monday, data, provided, by, daren, override ... (40472 more) ]

In [14]:
static Random rng = new Random(420);

static void Shuffle<T>(this IList<T> list)  
{  
    int n = list.Count;  
    while (n > 1) {  
        n--;  
        int k = rng.Next(n + 1);  
        T value = list[k];  
        list[k] = list[n];  
        list[n] = value;  
    }  
}

rawData.Shuffle();
var splitIndex = (int)(rawData.Length * 0.8);

var train = rawData.Take(splitIndex).ToArray();
var test = rawData.Skip(splitIndex).ToArray();

(train.Length + test.Length, rawData.Length)

Item1,5171
Item2,5171


In [15]:
class WordCounter {

    public int Count {get; private set;} = 0;
    public int Spam {get; private set;} = 0;
    public int Ham => Count - Spam;

    public void AddCount(bool spam) {
        Count++;
        if (spam)
            Spam++;
    }

    public override string ToString() => $"({Count} / {Spam} / {Ham})";
}

Dictionary<string, WordCounter> counters = bagOfWords.ToDictionary(x => x, _ => new WordCounter());

foreach (var r in train) {
    foreach (var word in r.Words) counters[word].AddCount(r.Spam);
}

counters.Take(5)

index value 0 [enron, (4100 / 0 / 4100)] Key enron Value (4100 / 0 / 4100) Count 4100 Spam 0 Ham 4100 1 [methanol, (62 / 0 / 62)] Key methanol Value (62 / 0 / 62) Count 62 Spam 0 Ham 62 2 [meter, (1692 / 0 / 1692)] Key meter Value (1692 / 0 / 1692) Count 1692 Spam 0 Ham 1692 3 [is, (5217 / 1683 / 3534)] Key is Value (5217 / 1683 / 3534) Count 5217 Spam 1683 Ham 3534 4 [a, (7048 / 2665 / 4383)] Key a Value (7048 / 2665 / 4383) Count 7048 Spam 2665 Ham 4383

In [ ]:
double SpamCount = rawData.Count(x => x.Spam);
double HamCount = rawData.Count(x => !x.Spam);

// double PSpam(WordCounter wc) => (double)wc.Spam / (double)wc.Count;
// double PHam(WordCounter wc) => (double)wc.Ham / (double)wc.Count;

double PSpam(WordCounter wc) => (double)wc.Spam / (double)SpamCount;
double PHam(WordCounter wc) => (double)wc.Ham / (double)HamCount;

// double PSpamL(WordCounter wc, double m = 1) => ((double)wc.Spam + m * 1/(double)wc.Count) / ((double)wc.Count + m);
// double PHamL(WordCounter wc, double m = 1) => ((double)wc.Ham + m * 1/(double)wc.Count) / ((double)wc.Count + m);

// Laplace
double PSpamL(WordCounter wc, double m = 1) => ((double)wc.Spam + m * 1/(double)SpamCount) / (SpamCount + m * (double)wc.Count);
double PHamL(WordCounter wc, double m = 1) => ((double)wc.Ham + m * 1/(double)HamCount) / (HamCount + m * (double)wc.Count);

double PSpam(ICollection<SpamDataSet> ds) => (double)ds.Count(x => x.Spam) / (double)ds.Count;
double PHam(ICollection<SpamDataSet> ds) => (double)ds.Count(x => !x.Spam) / (double)ds.Count;

(PSpam(train), PHam(train))

Item1,0.28820116054158607
Item2,0.7117988394584139


In [17]:
var _PSpam = PSpam(train);
var _PHam = PHam(train);

bool ClassifySpam(string text) {

    var pSpam = _PSpam;
    var pHam = _PHam;

    foreach (var word in GetWords(text)) {
        pSpam *= PSpam(counters[word]);
        pHam *= PHam(counters[word]);
    }

    return pSpam > pHam;
}

var check1 = test.Select(x => (x, ClassifySpam(x.Text))).ToArray();
var check2 = train.Select(x => (x, ClassifySpam(x.Text))).ToArray();

var correct = check1.Count(x => x.Item1.Spam == x.Item2);
var correct2 = check2.Count(x => x.Item1.Spam == x.Item2);

((double)correct / (double)check1.Length, (double)correct2 / (double)check2.Length)

Item1,0.744927536231884
Item2,0.9429400386847195


In [24]:
var _PSpam = PSpam(train);
var _PHam = PHam(train);

bool ClassifySpamL(string text) {

    var pSpam = _PSpam;
    var pHam = _PHam;

    foreach (var word in GetWords(text)) {
        pSpam *= PSpamL(counters[word]);
        pHam *= PHamL(counters[word]);
    }

    return pSpam > pHam;
}

var check1 = test.Select(x => (x, ClassifySpamL(x.Text))).ToArray();

var correct = check1.Count(x => x.Item1.Spam == x.Item2);
(double)correct / (double)check1.Length

0.914975845410628

In [19]:
var _PSpam = PSpam(train);
var _PHam = PHam(train);

bool ClassifySpamLog(string text) {

    var pSpam = Math.Log(_PSpam);
    var pHam = Math.Log(_PHam);

    foreach (var word in GetWords(text)) {
        pSpam += Math.Log(PSpam(counters[word]));
        pHam += Math.Log(PHam(counters[word]));
    }

    return pSpam > pHam;
}

var check1 = test.Select(x => (x, ClassifySpamLog(x.Text))).ToArray();

var correct = check1.Count(x => x.Item1.Spam == x.Item2);
(double)correct / (double)check1.Length

0.7468599033816425

In [20]:
var _PSpam = PSpam(train);
var _PHam = PHam(train);

bool ClassifySpamLogL(string text) {

    var pSpam = Math.Log(_PSpam);
    var pHam = Math.Log(_PHam);

    foreach (var word in GetWords(text)) {
        pSpam += Math.Log(PSpamL(counters[word]));
        pHam += Math.Log(PHamL(counters[word]));
    }

    return pSpam > pHam;
}

var check1 = test.Select(x => (x, ClassifySpamL(x.Text))).ToArray();

var correct = check1.Count(x => x.Item1.Spam == x.Item2);
(double)correct / (double)check1.Length

0.9120772946859903

# Post mortem

Jede Person beschreibt in der ILIAS-Abgabe individuell(!) die Bearbeitung des jeweiligen Aufgabenblattes
zurückblickend mit ca. 200 bis 400 Wörtern. Gehen Sie dabei aussagekräftig und nachvollziehbar auf folgende Punkte ein: 
 (a) Zusammenfassung: Was wurde gemacht?  
 (b) Implementierungsdetails: Kurze Beschreibung besonders interessanter Aspekte der Umsetzung.  
 (c) Was war der schwierigste Teil bei der Bearbeitung? Wie haben Sie dieses Problem gelöst?  
 (d) Was haben Sie gelernt oder (besser) verstanden?  
 (e) Team: Mit wem haben Sie zusammengearbeitet?  
 (f) Link zum Repo mit der Lösung  



 * a: Für die erste Aufgabe habe ich einen Naive Bayes für einen Datensatz aus einem vorherigen Praktikum schriftlich berechnet. Für die zweite Aufgabe habe ich einen Naive Bayes in C# implementiert, um einen E-Mail spamfilter zu erstellen. Die Textqualifikation wurde an einem Beispieldatensatz trainiert.
 * b: Für die Spam-Erkennung habe ich zu erst aus den Texten die Merkmale erzeugt. Dies habe ich sehr simple gehalten. Ich habe den Text an den Leerzeichen gesplittet und nur die Worte behalten, die Ausschließlich aus Buchstaben bestehen. Aus den Worten aus allen Texten einen "Bag of Words" erzeugt. Danach habe ich gezählt, wie häufig welche Worte in welcher Klasse vorkommen. Anhand der Anzahl der Vorkommnisse kann dan die Wahrscheinlichkeit berechnet werden, mit der ein Wort in einer Klasse vor kommt.
 * c: Ich hatte zu Anfang keinen Blick, wie ich den Naive Bayes überhaupt berechne. Ich habe dan die wichtigen Regeln Aufgeschrieben und mir Kommentare dazu geschrieben. Dadurch habe ich tatsächlich verstanden, wie der Algorithmus funktioniert und das er eigentlich recht einfach ist.
 * d: 
 * e: -
 * f: https://github.com/co1inco/IFM_Programieren3/tree/master/KI/10